In [ ]:
import sys
import os
import json
import time
from typing import Dict, List, Optional, Any
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# SPARQL and web requests
import requests
from SPARQLWrapper import SPARQLWrapper, JSON, POST, GET
from urllib.parse import quote_plus, urlencode
import rdflib

# EBRAINS SDK
try:
    from ebrains_kg_core.client import KGv3Client
    EBRAINS_SDK_AVAILABLE = True
except ImportError:
    print("EBRAINS SDK not available. Using REST API instead.")
    EBRAINS_SDK_AVAILABLE = False

# Local modules
from config import *
from minds_queries import QUERY_TEMPLATES

# Jupyter display
from IPython.display import display, HTML, JSON as DisplayJSON
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed

print("All dependencies loaded successfully!")


In [ ]:
class EBRAINSAuthenticator:
    """Handle EBRAINS authentication and token management"""
    
    def __init__(self):
        self.token = None
        self.client = None
        
    def setup_authentication(self, token: str = None):
        """
        Setup EBRAINS authentication
        
        Args:
            token: EBRAINS API token (optional if set in environment)
        """
        if token:
            self.token = token
        elif EBRAINS_TOKEN:
            self.token = EBRAINS_TOKEN
        else:
            print("⚠️  No EBRAINS token provided.")
            print("To get full access:")
            print("1. Register at: https://ebrains.eu/register")
            print("2. Generate token at: https://ebrains.eu/page/profile")
            print("3. Set token: auth.setup_authentication('your_token_here')")
            return False
            
        # Test authentication
        if self.test_connection():
            print("✅ EBRAINS authentication successful!")
            
            if EBRAINS_SDK_AVAILABLE:
                self.client = KGv3Client(token=self.token)
                print("✅ EBRAINS SDK client initialized")
            return True
        else:
            print("❌ Authentication failed. Please check your token.")
            return False
    
    def test_connection(self) -> bool:
        """Test EBRAINS API connection"""
        if not self.token:
            return False
            
        headers = {'Authorization': f'Bearer {self.token}'}
        try:
            response = requests.get(
                f"{EBRAINS_KG_API_V3}/types", 
                headers=headers,
                timeout=10
            )
            return response.status_code == 200
        except:
            return False
            
# Initialize authenticator
auth = EBRAINSAuthenticator()

# Interactive authentication setup
print("🔐 EBRAINS Authentication Setup")
print("=" * 40)

# For demo purposes, we'll also show public access methods
auth.setup_authentication()



In [ ]:
class MindsDataQuerier:
    """Execute SPARQL queries against EBRAINS Knowledge Graph"""
    
    def __init__(self, authenticator: EBRAINSAuthenticator):
        self.auth = authenticator
        self.base_url = EBRAINS_KG_API_V3
        
    def execute_sparql(self, query: str, limit: int = None) -> List[Dict]:
        """
        Execute SPARQL query against EBRAINS KG
        
        Args:
            query: SPARQL query string
            limit: Maximum number of results
            
        Returns:
            List of result dictionaries
        """
        if limit and 'LIMIT' not in query.upper():
            query += f'\nLIMIT {limit}'
            
        # Prepare request
        endpoint = f"{self.base_url}/queries"
        headers = {
            'Content-Type': 'application/json',
            'Accept': 'application/json'
        }
        
        if self.auth.token:
            headers['Authorization'] = f'Bearer {self.auth.token}'
            
        payload = {
            'query': query,
            'vocab': 'https://openminds.ebrains.eu/vocab/'
        }
        
        try:
            response = requests.post(
                endpoint, 
                json=payload, 
                headers=headers,
                timeout=TIMEOUT_SECONDS
            )
            
            if response.status_code == 200:
                data = response.json()
                return self._process_sparql_results(data)
            else:
                print(f"❌ Query failed with status {response.status_code}")
                print(f"Response: {response.text[:200]}...")
                return []
                
        except requests.exceptions.RequestException as e:
            print(f"❌ Network error: {e}")
            return []
    
    def _process_sparql_results(self, raw_data: Dict) -> List[Dict]:
        """Process raw SPARQL results into clean format"""
        if 'results' not in raw_data or 'bindings' not in raw_data['results']:
            return []
            
        results = []
        for binding in raw_data['results']['bindings']:
            result = {}
            for var, value_obj in binding.items():
                if 'value' in value_obj:
                    result[var] = value_obj['value']
                else:
                    result[var] = str(value_obj)
            results.append(result)
            
        return results
    
    def query_template(self, template_name: str, **kwargs) -> pd.DataFrame:
        """
        Execute a predefined query template
        
        Args:
            template_name: Name of query template
            **kwargs: Template parameters
            
        Returns:
            DataFrame with results
        """
        if template_name not in QUERY_TEMPLATES:
            print(f"❌ Template '{template_name}' not found")
            print(f"Available templates: {list(QUERY_TEMPLATES.keys())}")
            return pd.DataFrame()
            
        query = QUERY_TEMPLATES[template_name]
        
        # Simple template substitution
        for key, value in kwargs.items():
            query = query.replace(f'{{{key}}}', str(value))
            
        results = self.execute_sparql(query)
        return pd.DataFrame(results)

# Initialize querier
querier = MindsDataQuerier(auth)

print("🔍 MINDS Data Querier initialized!")
print("Available query templates:", list(QUERY_TEMPLATES.keys()))


In [ ]:
class PublicMindsAccess:
    """Access public MINDS data via EBRAINS Search API"""
    
    def __init__(self):
        self.search_url = EBRAINS_SEARCH_URL
        
    def search_datasets(self, 
                       query: str = "minds", 
                       dataset_type: str = None,
                       species: str = None,
                       size: int = 20) -> pd.DataFrame:
        """
        Search for datasets using public EBRAINS Search API
        
        Args:
            query: Search terms
            dataset_type: Filter by dataset type
            species: Filter by species
            size: Number of results
            
        Returns:
            DataFrame with search results
        """
        params = {
            'q': query,
            'type': 'Dataset',
            'size': size
        }
        
        if dataset_type:
            params['category'] = dataset_type
            
        if species:
            params['species'] = species
            
        try:
            response = requests.get(
                f"{self.search_url}/api/search",
                params=params,
                timeout=TIMEOUT_SECONDS
            )
            
            if response.status_code == 200:
                data = response.json()
                return self._process_search_results(data)
            else:
                print(f"❌ Search failed: {response.status_code}")
                return pd.DataFrame()
                
        except Exception as e:
            print(f"❌ Search error: {e}")
            return pd.DataFrame()
    
    def _process_search_results(self, data: Dict) -> pd.DataFrame:
        """Process search API results"""
        if 'hits' not in data or 'hits' not in data['hits']:
            return pd.DataFrame()
            
        results = []
        for hit in data['hits']['hits']:
            source = hit.get('_source', {})
            result = {
                'id': hit.get('_id', ''),
                'title': source.get('title', ''),
                'description': source.get('description', '')[:200],
                'type': source.get('type', ''),
                'species': ', '.join(source.get('species', [])),
                'techniques': ', '.join(source.get('techniques', [])),
                'contributors': ', '.join([c.get('name', '') for c in source.get('contributors', [])])
            }
            results.append(result)
            
        return pd.DataFrame(results)
    
    def get_dataset_details(self, dataset_id: str) -> Dict:
        """Get detailed information about a specific dataset"""
        try:
            response = requests.get(
                f"{self.search_url}/api/datasets/{dataset_id}",
                timeout=TIMEOUT_SECONDS
            )
            
            if response.status_code == 200:
                return response.json()
            else:
                return {}
                
        except Exception as e:
            print(f"❌ Error getting dataset details: {e}")
            return {}

# Initialize public access
public_access = PublicMindsAccess()

print("🌐 Public MINDS data access initialized!")


In [ ]:
print("🔍 MINDS Data Discovery Examples")
print("=" * 40)

# Example 1: Basic MINDS dataset search
print("\n1️⃣ Searching for MINDS datasets...")
minds_datasets = public_access.search_datasets("MINDS", size=10)

if not minds_datasets.empty:
    print(f"Found {len(minds_datasets)} datasets")
    display(minds_datasets[['title', 'type', 'species']].head())
else:
    print("No results from public search. Trying SPARQL query...")
    
    # Fallback to SPARQL
    sparql_results = querier.query_template('basic_minds')
    if not sparql_results.empty:
        print(f"Found {len(sparql_results)} datasets via SPARQL")
        display(sparql_results.head())
    else:
        print("Creating demo data for illustration...")
        demo_data = {
            'dataset': ['minds_001', 'minds_002', 'minds_003'],
            'name': ['Human Brain Atlas', 'Mouse Connectome', 'Primate Behavior'],
            'description': ['High-resolution human brain atlas', 'Mouse brain connectivity data', 'Behavioral analysis in primates']
        }
        minds_datasets = pd.DataFrame(demo_data)
        display(minds_datasets)

# Example 2: Species-specific search
print("\n2️⃣ Searching by species...")
species_results = public_access.search_datasets("", species="Homo sapiens", size=5)
if not species_results.empty:
    display(species_results[['title', 'species']].head())

# Example 3: Technique-specific search  
print("\n3️⃣ Searching by technique...")
technique_results = public_access.search_datasets("electrophysiology", size=5)
if not technique_results.empty:
    display(technique_results[['title', 'techniques']].head())


In [ ]:
print("🎯 Advanced SPARQL Query Examples")
print("=" * 40)

# Execute multiple query templates
query_results = {}

for template_name, description in [
    ('by_species', 'Datasets by Species'),
    ('spatial', 'Datasets with Spatial Information'), 
    ('temporal', 'Temporal/Longitudinal Datasets'),
    ('software', 'MINDS-related Software')
]:
    print(f"\n🔍 {description}")
    
    try:
        df = querier.query_template(template_name)
        if not df.empty:
            query_results[template_name] = df
            print(f"   Found {len(df)} results")
            display(df.head(3))
        else:
            print("   No results found")
    except Exception as e:
        print(f"   Error: {e}")


In [ ]:
print("📊 MINDS Data Visualization")
print("=" * 30)

def create_demo_data():
    """Create demonstration data for visualization"""
    return {
        'species': pd.DataFrame({
            'Species': ['Homo sapiens', 'Mus musculus', 'Rattus norvegicus', 'Macaca mulatta'],
            'Count': [45, 78, 32, 23],
            'Percentage': [25.3, 43.8, 18.0, 12.9]
        }),
        'techniques': pd.DataFrame({
            'Technique': ['Electrophysiology', 'Neuroimaging', 'Microscopy', 'Behavioral', 'Molecular'],
            'Count': [89, 67, 45, 34, 28],
            'Avg_Size_GB': [2.3, 15.7, 8.2, 0.8, 1.2]
        }),
        'temporal': pd.DataFrame({
            'Year': [2018, 2019, 2020, 2021, 2022, 2023, 2024],
            'Datasets': [12, 18, 25, 34, 41, 38, 29],
            'Cumulative': [12, 30, 55, 89, 130, 168, 197]
        })
    }

# Use real data if available, otherwise demo data
viz_data = create_demo_data()

# Update with real data if we have query results
if 'by_species' in query_results and not query_results['by_species'].empty:
    species_counts = query_results['by_species']['speciesName'].value_counts()
    viz_data['species'] = pd.DataFrame({
        'Species': species_counts.index,
        'Count': species_counts.values,
        'Percentage': (species_counts.values / species_counts.sum() * 100).round(1)
    })

# Create visualizations
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Species Distribution', 'Techniques Used', 'Dataset Growth', 'Data Size by Technique'),
    specs=[[{"type": "pie"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "bar"}]]
)

# Species pie chart
fig.add_trace(
    go.Pie(
        labels=viz_data['species']['Species'],
        values=viz_data['species']['Count'],
        name="Species"
    ),
    row=1, col=1
)

# Techniques bar chart
fig.add_trace(
    go.Bar(
        x=viz_data['techniques']['Technique'],
        y=viz_data['techniques']['Count'],
        name="Techniques",
        marker_color='lightblue'
    ),
    row=1, col=2
)

# Temporal growth line chart
fig.add_trace(
    go.Scatter(
        x=viz_data['temporal']['Year'],
        y=viz_data['temporal']['Datasets'],
        mode='lines+markers',
        name="Annual Datasets",
        line=dict(color='green')
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=viz_data['temporal']['Year'],
        y=viz_data['temporal']['Cumulative'],
        mode='lines+markers',
        name="Cumulative",
        line=dict(color='orange'),
        yaxis='y2'
    ),
    row=2, col=1
)

# Data size bar chart
fig.add_trace(
    go.Bar(
        x=viz_data['techniques']['Technique'],
        y=viz_data['techniques']['Avg_Size_GB'],
        name="Avg Size (GB)",
        marker_color='coral'
    ),
    row=2, col=2
)

# Update layout
fig.update_layout(
    height=800,
    showlegend=True,
    title_text="MINDS Data Analytics Dashboard",
    title_x=0.5
)

# Show the plot
fig.show()

# Summary statistics
print("\n📈 Summary Statistics:")
print(f"• Total datasets analyzed: {viz_data['species']['Count'].sum()}")
print(f"• Most common species: {viz_data['species'].iloc[0]['Species']} ({viz_data['species'].iloc[0]['Percentage']}%)")
print(f"• Most used technique: {viz_data['techniques'].iloc[0]['Technique']} ({viz_data['techniques'].iloc[0]['Count']} datasets)")
print(f"• Average data size: {viz_data['techniques']['Avg_Size_GB'].mean():.1f} GB")


In [ ]:
print("🔄 Interactive MINDS Data Explorer")
print("=" * 35)

def create_interactive_explorer():
    """Create interactive widgets for data exploration"""
    
    # Widget definitions
    query_type = widgets.Dropdown(
        options=[
            ('Basic MINDS Search', 'basic'),
            ('By Species', 'species'),
            ('By Technique', 'technique'), 
            ('Spatial Data', 'spatial'),
            ('Recent Data', 'recent')
        ],
        value='basic',
        description='Query Type:'
    )
    
    species_filter = widgets.Dropdown(
        options=['All'] + SPECIES_FILTERS,
        value='All',
        description='Species:'
    )
    
    technique_filter = widgets.Dropdown(
        options=['All'] + TECHNIQUE_FILTERS,
        value='All', 
        description='Technique:'
    )
    
    limit_slider = widgets.IntSlider(
        value=10,
        min=5,
        max=50,
        step=5,
        description='Results:'
    )
    
    search_button = widgets.Button(
        description='Search MINDS Data',
        button_style='primary',
        icon='search'
    )
    
    output_area = widgets.Output()
    
    def on_search_click(b):
        """Handle search button click"""
        with output_area:
            output_area.clear_output()
            print("🔍 Searching MINDS data...")
            
            # Build search parameters
            search_params = {
                'query_type': query_type.value,
                'species': species_filter.value if species_filter.value != 'All' else None,
                'technique': technique_filter.value if technique_filter.value != 'All' else None,
                'limit': limit_slider.value
            }
            
            # Execute search based on type
            try:
                if search_params['query_type'] == 'basic':
                    results = public_access.search_datasets("MINDS", size=search_params['limit'])
                elif search_params['query_type'] == 'species':
                    species_query = search_params['species'] or 'Homo sapiens'
                    results = public_access.search_datasets("", species=species_query, size=search_params['limit'])
                else:
                    # Use SPARQL for other queries
                    template_map = {
                        'spatial': 'spatial',
                        'recent': 'temporal',
                        'technique': 'basic_minds'
                    }
                    template = template_map.get(search_params['query_type'], 'basic_minds')
                    results = querier.query_template(template)
                    
                # Display results
                if isinstance(results, pd.DataFrame) and not results.empty:
                    print(f"✅ Found {len(results)} results")
                    display(results.head(search_params['limit']))
                    
                    # Create quick visualization
                    if len(results) > 3:
                        try:
                            if 'species' in results.columns:
                                species_counts = results['species'].value_counts().head(5)
                                plt.figure(figsize=(10, 4))
                                species_counts.plot(kind='bar')
                                plt.title('Top Species in Search Results')
                                plt.xticks(rotation=45)
                                plt.tight_layout()
                                plt.show()
                        except:
                            pass
                else:
                    print("❌ No results found with current parameters")
                    
            except Exception as e:
                print(f"❌ Search error: {e}")
    
    search_button.on_click(on_search_click)
    
    # Layout widgets
    controls = widgets.VBox([
        widgets.HTML("<h3>🔍 MINDS Data Search Interface</h3>"),
        query_type,
        widgets.HBox([species_filter, technique_filter]),
        limit_slider,
        search_button
    ])
    
    return widgets.VBox([controls, output_area])

# Create and display the interactive explorer
explorer = create_interactive_explorer()
display(explorer)


In [ ]:
print("🔗 MINDS Data Integration Examples")
print("=" * 38)

def demonstrate_data_integration():
    """Show how MINDS data integrates with other neuroscience resources"""
    
    print("1️⃣ MINDS + Brain Atlases Integration")
    print("-" * 40)
    
    # Example: Link MINDS datasets with brain atlas regions
    integration_example = """
    PREFIX openminds: <https://openminds.ebrains.eu/vocab/>
    PREFIX sands: <https://openminds.ebrains.eu/sands/>
    
    SELECT ?dataset ?atlas ?region ?coordinates
    WHERE {
        ?dataset a openminds:Dataset ;
                 openminds:spatialLocation ?location .
        ?location sands:atlas ?atlas ;
                 sands:brainRegion ?region ;
                 sands:coordinates ?coordinates .
        FILTER(CONTAINS(LCASE(str(?dataset)), "minds"))
    }
    """
    
    print("Example SPARQL query for spatial integration:")
    print(integration_example)
    
    print("\n2️⃣ MINDS + Neuroshapes Schema Validation")
    print("-" * 45)
    
    validation_example = """
    # Python code to validate MINDS data against neuroshapes
    from rdflib import Graph
    
    def validate_against_neuroshapes(dataset_uri):
        # Load dataset RDF
        dataset_graph = Graph()
        dataset_graph.parse(dataset_uri)
        
        # Load neuroshapes schema
        schema_graph = Graph()
        schema_graph.parse("https://neuroshapes.org/schemas/dataset")
        
        # Perform validation
        # (This would use SHACL validation in practice)
        return validation_results
    """
    
    print("Python integration example:")
    print(validation_example)
    
    print("\n3️⃣ Cross-Database Queries")
    print("-" * 28)
    
    federated_example = """
    # Federated query example combining MINDS + Wikidata
    SELECT ?dataset ?species ?wikidataInfo
    WHERE {
        # MINDS data
        ?dataset openminds:studiedSpecies ?species .
        
        # Link to external knowledge
        SERVICE <https://query.wikidata.org/sparql> {
            ?species rdfs:label ?wikidataInfo .
            FILTER(LANG(?wikidataInfo) = "en")
        }
    }
    """
    
    print("Federated query example:")
    print(federated_example)

demonstrate_data_integration()


In [ ]:
print("💡 Best Practices for MINDS Data Access")
print("=" * 42)

best_practices = {
    "Authentication": [
        "Always use secure token storage",
        "Refresh tokens regularly", 
        "Never commit tokens to version control",
        "Use environment variables for production"
    ],
    
    "Query Optimization": [
        "Use LIMIT clauses to avoid large result sets",
        "Filter early in your SPARQL queries",
        "Cache frequently used results",
        "Use specific property paths instead of wildcards"
    ],
    
    "Error Handling": [
        "Always wrap API calls in try-catch blocks",
        "Implement exponential backoff for retries",
        "Log errors for debugging",
        "Provide fallback options for users"
    ],
    
    "Data Processing": [
        "Validate data before processing",
        "Handle missing values gracefully",
        "Use appropriate data types",
        "Document your data transformations"
    ],
    
    "Performance": [
        "Use pagination for large datasets",
        "Implement result caching",
        "Batch API calls when possible",
        "Monitor rate limits"
    ]
}

for category, practices in best_practices.items():
    print(f"\n {category}:")
    for i, practice in enumerate(practices, 1):
        print(f"   {i}. {practice}")


In [ ]:
print("\n🔧 Common Issues and Solutions")
print("=" * 35)

troubleshooting = {
    "Authentication Errors": {
        "Problem": "401 Unauthorized or 403 Forbidden",
        "Solutions": [
            "Check token validity and expiration",
            "Verify token permissions",
            "Ensure correct Authorization header format"
        ]
    },
    
    "Query Timeouts": {
        "Problem": "Queries taking too long or timing out",
        "Solutions": [
            "Add LIMIT clauses to queries",
            "Optimize query structure",
            "Use more specific filters",
            "Break complex queries into smaller parts"
        ]
    },
    
    "Empty Results": {
        "Problem": "Queries return no data",
        "Solutions": [
            "Check query syntax and semantics",
            "Verify property URIs and namespaces", 
            "Start with broader queries and narrow down",
            "Check data availability in target endpoints"
        ]
    },
    
    "Network Issues": {
        "Problem": "Connection errors or slow responses",
        "Solutions": [
            "Check internet connectivity",
            "Verify endpoint URLs",
            "Implement retry logic",
            "Use appropriate timeout settings"
        ]
    }
}

for issue, details in troubleshooting.items():
    print(f"\n {issue}")
    print(f"   Problem: {details['Problem']}")
    print("   Solutions:")
    for i, solution in enumerate(details['Solutions'], 1):
        print(f"      {i}. {solution}")


In [ ]:
print("\n Next Steps and Additional Resources")
print("=" * 42)

resources = {
    "EBRAINS Platform": [
        "Main portal: https://ebrains.eu/",
        "Data search: https://search.kg.ebrains.eu/",
        "Documentation: https://docs.ebrains.eu/",
        "Knowledge Graph: https://kg.ebrains.eu/"
    ],
    
    "SPARQL Learning": [
        "W3C SPARQL Tutorial: https://www.w3.org/TR/sparql11-query/",
        "SPARQL by Example: https://www.cambridge.org/core/books/learning-sparql/",
        "Interactive SPARQL: https://query.wikidata.org/",
        "SPARQL Playground: https://yasgui.triply.cc/"
    ],
    
    "Neuroscience Standards": [
        "Neuroshapes: https://neuroshapes.org/",
        "BIDS: https://bids.neuroimaging.io/",
        "NIDM: http://nidm.nidash.org/",
        "FAIR principles: https://www.go-fair.org/fair-principles/"
    ],
    
    "Development Tools": [
        "EBRAINS SDK: https://ebrains-kg-core.readthedocs.io/",
        "RDFLib: https://rdflib.readthedocs.io/",
        "SPARQLWrapper: https://sparqlwrapper.readthedocs.io/",
        "Jupyter Notebooks: https://jupyter.org/"
    ]
}

for category, links in resources.items():
    print(f"\n📚 {category}:")
    for link in links:
        print(f"   • {link}")

print("\n" + "="*60)
print("🎉 Tutorial Complete!")
print("You now have comprehensive access to MINDS data through")
print("SPARQL queries and REST APIs. Happy data exploration!")
print("="*60)